In [ ]:
# import libraries

import pandas as pd
import numpy as np
import seaborn as sns
sns.set(color_codes=True)
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
import math 

#update this path with the path where you stored the datasets
data_path = "/data/test_newrepo" 

In [ ]:
#Load LUTs

lut_name_soft = "soft_lut"
lut_name_medium = "medium_lut"
lut_name_hard = "hard_lut"

lut_soft = np.load(data_path+'/LUT_'+lut_name_soft+'.npy')
lut_medium = np.load(data_path+'/LUT_'+lut_name_medium+'.npy')
lut_hard = np.load(data_path+'/LUT_'+lut_name_hard+'.npy')

In [ ]:
lut_medium.shape

In [ ]:
#test with LUTs
run_name_test = "dataset"
file_name_test = "run63_medium_10fact_mega_shared"

file_path = data_path+'/'+file_name_test+'_dataset.pkl'

# Load the array from the pickle file
with open(file_path, 'rb') as file:
    loaded_array_test = pickle.load(file)


In [ ]:
loaded_array_test.shape

In [ ]:
# Use these parameters to filter the dataset. Put them equal to 0 to use all the dataset.

filter_flux = 0
filter_spectra = 0
filter_theta=0
filter_phi=0

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_flux == 1:
    for grb in loaded_array_test:
        if grb['flux'] >1 and grb['flux'] <= 1.6:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

print(len(loaded_array_test))
filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_spectra==1:
    for grb in loaded_array_test:
        if  "1500" in grb['spectrum']: #["Band 10 10000 -1.9 -3.7 230","Band 10 10000 -1 -2.3 699.9","Comptonized 10 10000 -0.5 1500"]
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

print(len(loaded_array_test))
count = 0
filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

if filter_theta==1:
    print("filter spectra")
    for grb in loaded_array_test:
        if float(grb['coord'][0])>50 and float(grb['coord'][0])<130:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

count = 0
filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

if filter_phi==1:
    print("filter spectra")
    for grb in loaded_array_test:
        if float(grb['coord'][1])>100 and float(grb['coord'][1])<170:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

In [ ]:
test_dataset = loaded_array_test
print(test_dataset.shape)

In [ ]:
def get_radians(coords):
    # Unpack the list of (theta, phi) pairs
    theta, phi = zip(*coords)
    
    # Convert to numpy arrays
    theta = np.array(theta)
    phi = np.array(phi)
    
    # Wrap phi values >180 into the range (-180, 180]
    mask = phi > 180
    phi[mask] -= 360
    
    # Convert theta into colatitude (90 - theta)
    theta = 90 - theta

    # Return values in radians
    return np.radians(theta), np.radians(phi)

def calculate_chi_squared_optimized(s, b, m):
    """
    Compute the chi-squared value for each position in the grid in an optimized way.

    Parameters:
        s: array of shape (6,) with observed counts (s(j)).
        b: array of shape (12,) with background counts (b(j)).
        m: array of shape (12, 41168) with model counts (m(j, i)).

    Returns:
        chi_squared: array of shape (41168,) with the chi-squared value for each position i.
    """
    # Use only the first 6 detectors
    indices = np.arange(6)
    
    # Extract model counts for these detectors
    m_subset = m[:, indices]  # Shape: (41168, 6)

    # Compute numerator and denominator for normalization factor f_i
    with np.errstate(divide='ignore', invalid='ignore'):
        numerator = np.sum(m_subset * (s[indices] - b[indices]) / s[indices], axis=1)
        denominator = np.sum((m_subset**2) / s[indices], axis=1)

        # Avoid division by zero
        f_i = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator != 0)

        # Expand f_i for broadcasting
        f_i_expanded = f_i[:, np.newaxis]  # Shape: (41168, 1)

        # Compute chi-squared terms
        chi_numerator = (s[indices] - b[indices] - f_i_expanded * m_subset)**2
        chi_denominator = b[indices] + f_i_expanded * m_subset

        # Safe division for chi-squared elements
        chi_squared_elements = np.divide(
            chi_numerator, chi_denominator,
            out=np.full_like(chi_numerator, np.finfo(np.float64).max),
            where=chi_denominator != 0
        )

    # Sum over detectors to obtain chi^2 per direction
    chi_squared = np.sum(chi_squared_elements, axis=1)

    return chi_squared


def angular_distance(theta1, phi1, theta2, phi2):
    """
    Compute the angular (great-circle) distance in degrees between two points
    specified by (theta, phi) coordinates in degrees.

    Parameters:
        theta1, phi1: floats, coordinates of the first point (degrees).
        theta2, phi2: floats, coordinates of the second point (degrees).

    Returns:
        angular_dist_deg: float, angular distance between the two points in degrees.
    """
    # Adjust theta values by shifting -90
    theta1 = 90 - theta1
    theta2 = 90 - theta2

    # Convert angles to radians
    theta1_rad = math.radians(theta1)
    phi1_rad = math.radians(phi1)
    theta2_rad = math.radians(theta2)
    phi2_rad = math.radians(phi2)
    
    # Compute the difference in longitude
    delta_phi = abs(phi1_rad - phi2_rad)

    # Apply spherical law of cosines
    value = (math.sin(theta1_rad) * math.sin(theta2_rad) +
             math.cos(theta1_rad) * math.cos(theta2_rad) * math.cos(delta_phi))
    
    # Clamp value to [-1, 1] to avoid floating point errors in acos
    value_clamped = max(-1.0, min(1.0, value)) 
    
    angular_dist = math.acos(value_clamped)
    
    # Convert angular distance from radians to degrees
    angular_dist_deg = math.degrees(angular_dist)
    
    return angular_dist_deg

def diff_phi(a1, a2):
    diff = abs(a1 - a2)
    if diff > 180:
        diff = 360 - diff
    return diff

In [ ]:
def find_closest_pixel(theta_real, phi_real, best_lut):
    # Convert input angles from degrees to radians
    theta_real_rad = np.deg2rad(theta_real)
    phi_real_rad = np.deg2rad(phi_real)
    
    # Extract θ and φ from best_lut (last two columns, in degrees) and convert to radians
    theta_vals_m = np.deg2rad(best_lut[:, -2])
    phi_vals_m = np.deg2rad(best_lut[:, -1])
    
    # Compute the cosine of the spherical angular distance
    cos_dist = (
        np.sin(theta_real_rad) * np.sin(theta_vals_m) * np.cos(phi_real_rad - phi_vals_m)
        + np.cos(theta_real_rad) * np.cos(theta_vals_m)
    )

    # Compute the angular distance (in radians)
    angular_distance = np.arccos(cos_dist)
    
    # Find the index of the minimum distance
    min_idx = np.argmin(angular_distance)
    
    # The index of the maximum cosine corresponds to the minimum distance
    # min_idx = np.argmax(cos_dist)
    return min_idx

In [ ]:
import healpy as hp
import numpy as np
from scipy.stats import chi2

def analyze_grbs(test_dataset,with_background):

    results = []
    spectra_fitted = 0
    good_spectra_fit_distances = []
    bad_spectra_fit_distances=[]
    
    # Background counts for the 6 detectors found in DC3 background data
    b_sim = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617])

    count = 0
    for grb_list in test_dataset:

        dist_list = []
        area_list = []
        
        for grb in grb_list:
    
            count = count +1
    
            if count % 1000 == 0:
                print(count)
            
            counts = grb['counts']
            spectrum = grb['spectrum']
            if "230" in spectrum :
                spectra_value="soft"
            elif "699.9" in spectrum:
                spectra_value="medium"
            elif "Compton" in spectrum:
                spectra_value="hard"
            else:
                spectra_value="random"       
            
            theta_real = float(grb['coord'][0])
            phi_real = float(grb['coord'][1])

            
            
            if with_background==1:            
            
                rand_bkg = np.random.poisson(b_sim*20)
                chi_squared_medium = calculate_chi_squared_optimized(np.array(counts)+rand_bkg,b_sim*20, lut_medium)
                min_medium = np.min(chi_squared_medium)
    
            elif with_background==0:
                
                chi_squared_medium = calculate_chi_squared_optimized(np.array(counts), np.array([0,0,0,0,0,0]), lut_medium)
                min_medium = np.min(chi_squared_medium)
                
            elif with_background==2:
            
                chi_squared_medium = calculate_chi_squared_optimized(np.array(lut_medium[count][:6])+b_sim*20,b_sim*20, lut_medium)
                min_medium = np.min(chi_squared_medium)
        
        
            global_min = min_medium 
           
            spectra_selected = False
            
            
            if global_min == min_medium:
                chi2_array = chi_squared_medium
                if spectra_value == "medium":
                    spectra_fitted = spectra_fitted +1
                    spectra_selected = True
                argmin_index = np.argmin(chi_squared_medium)
                best_lut = lut_medium
           
            
            theta_loc = best_lut[argmin_index][6]
            phi_loc = best_lut[argmin_index][7]
        
            theta_real = float(grb['coord'][0])
            phi_real = float(grb['coord'][1])
    
            dist = angular_distance(theta_loc,phi_loc,theta_real,phi_real)
        
            if spectra_selected:
                good_spectra_fit_distances.append(dist)
            else:
                bad_spectra_fit_distances.append(dist)
                
            map = chi2_array
            limit = global_min+ chi2.ppf(0.9, 2)
            
            # Maschera i valori maggiori o uguali a X
            nside = hp.npix2nside(len(chi2_array))
            mappa_masked = np.ma.masked_where(map  >= limit , map)
            npix_in_region = np.sum(~mappa_masked.mask)
            pix_area = hp.nside2pixarea(nside, degrees=True) 
            area_90 = npix_in_region * pix_area
            
            dist_list.append(dist)
            area_list.append(area_90)

        distance = np.mean(dist_list)
        area = np.mean(area_list)
        
        results.append([distance,area])

    return results,spectra_fitted, good_spectra_fit_distances,bad_spectra_fit_distances

In [ ]:
# 0 = no background, 1 = poissonian background, 2 = use LUTs as source and add background

results,spectra_fitted, good_spectra_fit_distances,bad_spectra_fit_distances= analyze_grbs(test_dataset,0)

In [ ]:
distances = []
area_array = []

for r in results:
    distances.append(r[0])
    area_array.append(r[1])

distances = np.array(distances)
area_array = np.array(area_array)


In [ ]:
np.mean(distances)

In [ ]:
import pickle
if False:

    # Salva l'array in un file usando pickle
    with open(prefix+"/data/chi2_"+file_name_test+"_distall.pkl", "wb") as f:
        pickle.dump(distances, f)
        
    # Salva l'array in un file usando pickle
    with open(prefix+"/data/chi2_"+file_name_test+"_cont_area.pkl", "wb") as f:
        pickle.dump(area_array, f)



In [ ]:
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
import pickle
hp.projview(
    np.array(area_array),
    coord=["G"],
    projection_type="aitoff",          
    graticule=True,
    graticule_labels=True,
    longitude_grid_spacing=60,
    title=file,
    latitude_grid_spacing=30,
    cmap="turbo",
    nest=True,
    unit="", 
    fontsize={
        "xlabel": 14,
        "ylabel": 14,
        "title": 16,
        "xtick_label": 14,
        "ytick_label": 14,
        "cbar_label": 14,
        "cbar_tick_label": 14 
    },
    override_plot_properties={
        "cbar_shrink": 0.8,
        "cbar_pad": 0.05,
        "cbar_label_pad": 5
    }
    
)
plt.show()